# Segment scorecard evaluation

**Q1** -- is the pooled logistic score good on each business segment?  
**Q2** -- is a same-predictor WoE refit worth a split, or is intercept/slope recalibration enough?

This notebook runs top to bottom on synthetic data and a **fake in-memory database**, so it needs no
credentials and no real warehouse. It walks the full flow:

1. smart data creation -- resolve the analysis metadata from a table plus a few mandatory columns;
2. capability resolution -- what can and cannot be analysed, and why;
3. interactive confirmation and settings persistence (with the non-interactive escape hatch);
4. optimal WoE binning, the `cols_pred` → `cols_pred_woe` map, and a serialised `grouping.json`;
5. segment evaluation, including matched approval-rate Gini, predictor stability, grouping comparison
   and saved refit / recalibration artefacts;
6. a production logistic scorecard written as SQL -- columns, formula, grouping with null
   imputation, and a sklearn `LogisticRegression` that reproduces `PD = 1/(1+exp(-B^T X))`;
7. the final HTML report with prioritised recommendations.

The package targets Python 3.6+, so nothing here uses 3.7+ syntax.

In [1]:
import os
import warnings

import pandas as pd

from scorecard_segment_eval import (
    BinningModel,
    FakeSqlExecutor,
    Gates,
    action_list,
    available_queries,
    build_analysis_frame,
    confirm_settings,
    decision_table,
    evaluate_segments,
    load_fitted_artifact,
    load_settings,
    make_synthetic_db_table,
    map_pred_to_woe,
    parse_scorecard_sql_path,
    predictor_stability,
    recommendations,
    render_metadata_summary,
    resolve_metadata,
    save_grouping,
    save_report,
    stability_summary,
)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)


def _find_repo_root():
    here = os.path.abspath(os.getcwd())
    for candidate in (here, os.path.dirname(here)):
        if os.path.isdir(os.path.join(candidate, "scorecard_segment_eval")):
            return candidate
    return here


REPO_ROOT = _find_repo_root()
WORKDIR = os.path.join(REPO_ROOT, "notebooks", "demo_output")
SQL_PATH = os.path.join(REPO_ROOT, "tests", "fixtures", "sample_scorecard.sql")
if not os.path.isdir(WORKDIR):
    os.makedirs(WORKDIR)
print("artefacts will be written to", WORKDIR)
print("sample scorecard SQL:", SQL_PATH)

artefacts will be written to /workspace/notebooks/demo_output
sample scorecard SQL: /workspace/tests/fixtures/sample_scorecard.sql


## 1. Stand in for the database

`make_synthetic_db_table` returns the synthetic book with production-style column names
(`SKP_CREDIT_CASE`, `DATE_DECISION`, `PORTFOLIO`, `TargetA`, `TargetAObs`, ...). `FakeSqlExecutor`
answers the shipped `.sql` templates against it, so the code path is identical to a real warehouse:
an executor is just a callable that takes SQL and returns a DataFrame.

In [2]:
db_table = make_synthetic_db_table(n=12000, seed=7, portfolio="PortfolioA")
executor = FakeSqlExecutor(table=db_table)

print("query templates on disk:", available_queries())
db_table.head()

query templates on disk: ['column_profile', 'describe_table', 'fetch_columns_by_id', 'fetch_portfolio']


,SKP_CREDIT_CASE,DATE_DECISION,PORTFOLIO,TargetAObs,TargetA,PD,x1,x2,cat,x1_woe,FLAG_FANTOMAS,CHANNEL
0,0,2024-05-25,PortfolioA,1,0,0.445260,1.682289,-1.133449,A,1.682289,0,miscal
1,1,2023-12-04,PortfolioA,1,0,0.227366,0.077222,0.247662,C,0.077222,0,miscal
2,2,2024-01-05,PortfolioA,1,0,0.066946,-0.303873,-2.110806,B,-0.303873,0,miscal
3,3,2024-04-29,PortfolioA,1,0,0.103969,-0.218961,-1.568250,B,-0.218961,0,miscal
4,4,2023-11-09,PortfolioA,0,0,0.524895,-0.183471,0.988434,B,-0.183471,0,miscal


## 2. Smart data creation

Only three things are mandatory: the credit case id, the score column, and the predictors the pooled
model actually uses -- or a production scorecard SQL file, which fills `cols_pred_used` for you
(section 7). The date column, the segmentation columns and the target / observation-flag pair are
looked up from the database. **Every inference emits a warning**, so an inferred setting can never
be mistaken for one you supplied.

Here we deliberately withhold `cols_pred` and `cols_pred_woe` to see the capability report degrade.

In [3]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    minimal = resolve_metadata(
        table="risk.scorecard_base",
        col_id="SKP_CREDIT_CASE",
        col_score="PD",
        cols_pred_used=["x1_woe", "x2", "cat"],
        executor=executor,
    )

for warning in caught:
    print("WARNING:", warning.message)

In [4]:
print("SQL actually executed:", [q["query"] for q in executor.queries])
print()
print(executor.queries[-1]["sql"])

SQL actually executed: ['describe_table', 'column_profile', 'fetch_portfolio']

-- query: fetch_portfolio
-- Portfolio mix of the credit cases in the analysis table.  The dominant
-- portfolio drives the target / observation-flag auto-detection.
-- Parameters: table, col_id, col_portfolio, sample_limit
SELECT
    PORTFOLIO          AS portfolio,
    COUNT(DISTINCT SKP_CREDIT_CASE) AS n_cases
FROM (
    SELECT SKP_CREDIT_CASE, PORTFOLIO
    FROM risk.scorecard_base
    LIMIT 100000
) sampled
GROUP BY PORTFOLIO
ORDER BY n_cases DESC
-- params: {"col_id": "SKP_CREDIT_CASE", "col_portfolio": "PORTFOLIO", "sample_limit": 100000, "table": "risk.scorecard_base", "table_name": "scorecard_base", "table_schema": "risk"}



In [5]:
minimal.capabilities.to_frame()

,analysis,status,reason
0,segment_performance,available,
1,vintage_stability,available,
2,matched_ar_gini,available,
3,psi_numeric,available,
4,report,available,
5,psi_grouping,blocked,no usable grouping.json was supplied
6,recalibration,blocked,needs cols_pred_woe (the WoE inputs the pooled...
7,refit,blocked,needs cols_pred (not inferable) and col_date f...
8,submodel,blocked,"needs cols_pred, col_target, col_date and the ..."


Without `cols_pred` there is no refit and no sub-model; without `cols_pred_woe` there is no
recalibration. Supply them and those analyses unlock. We also point at a `grouping.json` and name the
fantomas flag. `ScorecardColumns.pred_woe_map()` then pairs each raw predictor with its WoE column
(`x1` → `x1_woe`; predictors with no counterpart are omitted).

A `grouping.json` normally comes out of model development, or is parsed from production scorecard SQL.
Here we create one from the development sample so the notebook is self-contained.

In [6]:
grouping_path = os.path.join(WORKDIR, "grouping.json")

development = db_table.loc[db_table["TargetAObs"].eq(1)]
grouping = BinningModel.fit(
    development[["x1", "x2", "cat"]], development["TargetA"], Gates()
)
save_grouping(grouping, grouping_path)
grouping.iv_table()

,feature,kind,method,n_bins,iv,monotonic,notes
0,x1,numeric,tree,4,0.108879,True,merged_monotonicity_violation
1,x2,numeric,tree,6,0.182589,True,merged_monotonicity_violation
2,cat,categorical,categorical_woe,3,0.018990,False,


The grouping is supervised, not equal-frequency: `optbinning` is used when installed, otherwise a
depth- and leaf-constrained decision tree supplies the cut points. Sparse bins are merged, missing
values get their own bin, and monotonicity is enforced only when it does not throw away the signal
(`monotonicity_not_feasible` in the notes means the relation is genuinely non-monotone).

In [7]:
spec = grouping.specs["x1"]
print("method: ", spec.method)
print("monotonic:", spec.monotonic)
print("notes:   ", spec.notes)
pd.DataFrame(
    {
        "bin": spec.bin_order(),
        "count": [spec.counts.get(b) for b in spec.bin_order()],
        "events": [spec.events.get(b) for b in spec.bin_order()],
        "woe": [spec.woe.get(b) for b in spec.bin_order()],
    }
)

method:  tree
monotonic: True
notes:    ['merged_monotonicity_violation']


,bin,count,events,woe
0,"(-inf, 0.544101]",7411.0,1180.0,0.189597
1,"(0.544101, 1.12006]",1682.0,351.0,-0.142230
2,"(1.12006, 1.63489]",888.0,221.0,-0.370965
3,"(1.63489, inf)",531.0,206.0,-1.019020
4,__missing__,0.0,0.0,0.000000
5,__other__,0.0,0.0,0.000000


In [8]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    meta = resolve_metadata(
        table="risk.scorecard_base",
        col_id="SKP_CREDIT_CASE",
        col_score="PD",
        cols_pred_used=["x1_woe", "x2", "cat"],
        cols_pred=["x1", "x2", "cat"],
        cols_pred_woe=["x1_woe"],
        col_fantomas="FLAG_FANTOMAS",
        grouping_path=grouping_path,
        executor=executor,
    )

print("inferred:", meta.inferred_fields())
print("supplied:", meta.supplied_fields())
print("missing: ", meta.missing_fields())
meta.capabilities.to_frame()

inferred: ['col_date', 'col_obs', 'col_target', 'cols_segment', 'portfolio']
supplied: ['col_fantomas', 'col_id', 'col_score', 'cols_pred', 'cols_pred_used', 'cols_pred_woe', 'grouping_path']
missing:  []


,analysis,status,reason
0,segment_performance,available,
1,vintage_stability,available,
2,matched_ar_gini,available,
3,psi_numeric,available,
4,psi_grouping,available,
5,recalibration,available,
6,refit,available,
7,report,available,
8,submodel,blocked,needs cols_pred plus the optional 'xgboost' extra


`map_pred_to_woe` uses a `_woe` suffix first, then a unique prefix / case / `woe_` affix match.
`ScorecardColumns.pred_woe_map()` is the same heuristic; an explicit `pred_map` from parsed SQL
wins when the alias does not follow the suffix convention (`indosat_v2` → `feature_a_WOE`).

In [9]:
print("heuristic:", map_pred_to_woe(["predA", "predB"], ["predA_woe"]))
print("from ScorecardColumns:", meta.columns.pred_woe_map())

heuristic: {'predA': 'predA_woe'}
from ScorecardColumns: {'x1': 'x1_woe'}


## 3. Confirm and persist the settings

`confirm_settings` prints the resolved metadata, the provenance of every field, the warnings and the
capability list, then asks for confirmation with `input()` and for a file name to save under.

Interactively you would just call `confirm_settings(meta)`. To keep this notebook runnable headlessly
we pass `auto_confirm=True` -- the same thing happens automatically when there is no TTY or when
`SCORECARD_EVAL_AUTO_CONFIRM=1` is set.

In [10]:
print(render_metadata_summary(meta))

Resolved analysis metadata
table: risk.scorecard_base
  col_id:          SKP_CREDIT_CASE                          [supplied]
  col_score:       PD                                       [supplied]
  col_date:        DATE_DECISION                            [inferred:catalogue_candidate]
  col_target:      TargetA                                  [inferred:portfolio=PortfolioA]
  col_obs:         TargetAObs                               [inferred:portfolio=PortfolioA]
  col_fantomas:    FLAG_FANTOMAS                            [supplied]
  cols_segment:    CHANNEL                                  [inferred:catalogue_candidate]
  cols_pred:       x1, x2, cat                              [supplied]
  cols_pred_woe:   x1_woe                                   [supplied]
  pred_woe_map:    x1 -> x1_woe; x2 -> (none); cat -> (none) [derived]
  cols_pred_used:  x1_woe, x2, cat                          [supplied]
  grouping_path:   /workspace/notebooks/demo_output/grouping.json [supplied]
  port

In [11]:
settings_path = os.path.join(WORKDIR, "scorecard_eval_settings.json")

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    confirmation = confirm_settings(
        meta,
        auto_confirm=True,  # drop this argument for the interactive y/N prompt
        settings_path=settings_path,
        display_fn=lambda text: None,  # already displayed above
    )

print("confirmed:      ", confirmation.confirmed)
print("auto-confirmed: ", confirmation.auto)
print("saved to:       ", confirmation.settings_path)

cached = load_settings(settings_path)
print("round-trip lossless:", cached.to_dict() == meta.to_dict())

confirmed:       True
auto-confirmed:  True
saved to:        /workspace/notebooks/demo_output/scorecard_eval_settings.json
round-trip lossless: True


### Interactive prompting

The prompt is injectable, which is how the tests drive it. In a real notebook session the two calls
below would show `Proceed with these settings? [y/N]:` and `Save settings as [...]:`.

In [12]:
scripted = iter(["y", os.path.join(WORKDIR, "chosen_name.json")])


def scripted_prompt(message):
    answer = next(scripted)
    print(message + answer)
    return answer


interactive = confirm_settings(
    meta, auto_confirm=False, prompt=scripted_prompt, display_fn=lambda text: None
)
print("confirmed:", interactive.confirmed, "| saved to:", interactive.settings_path)

Proceed with these settings? [y/N]: y
Save settings as [scorecard_eval_settings.json]: /workspace/notebooks/demo_output/chosen_name.json
confirmed: True | saved to: /workspace/notebooks/demo_output/chosen_name.json


## 4. Assemble the analysis frame

`build_analysis_frame` fetches only the columns the caller does not already have, merging on the
credit case id, and warns about everything it pulled.

In [13]:
local_extract = db_table[["SKP_CREDIT_CASE", "PD", "DATE_DECISION"]].copy()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    df = build_analysis_frame(meta, executor, base=local_extract)

for warning in caught:
    print("WARNING:", warning.message)
df.head()

,SKP_CREDIT_CASE,PD,DATE_DECISION,TargetA,TargetAObs,FLAG_FANTOMAS,CHANNEL,x1,x2,cat,x1_woe
0,0,0.445260,2024-05-25,0,1,0,miscal,1.682289,-1.133449,A,1.682289
1,1,0.227366,2023-12-04,0,1,0,miscal,0.077222,0.247662,C,0.077222
2,2,0.066946,2024-01-05,0,1,0,miscal,-0.303873,-2.110806,B,-0.303873
3,3,0.103969,2024-04-29,0,1,0,miscal,-0.218961,-1.568250,B,-0.218961
4,4,0.524895,2023-11-09,0,0,0,miscal,-0.183471,0.988434,B,-0.183471


## 5. Evaluate the segments

`n_jobs=-1` fans the per-segment work out across cores. Results are identical to `n_jobs=1`: every
bootstrap replicate is seeded from `(seed, replicate index)` rather than from shared global state.

In [14]:
gates = Gates(
    min_n=400,
    min_events=25,
    n_bootstrap=80,
    bootstrap_seed=0,
    holdout_frac=0.3,
    n_jobs=-1,
)

result = evaluate_segments(df, meta.columns, gates, grouping=grouping)
decision_table(result)

,segment_col,segment_value,important,q1_verdict,failed_pillars,q2_action,q2_reason,gini,gini_ratio,oe,ece,volume_share,default_share,ar_gap,gini_at_matched_ar,ar_artifact_suspected,psi_method,stability_score,unstable_predictors
0,CHANNEL,core,True,GOOD,,KEEP_POOLED,performance_good,0.524566,2.398196,1.127250,0.019669,0.593333,0.554137,0.046759,NaN,False,grouping_bins,NaN,
1,CHANNEL,inverted,True,WEAK,"rank_order,calibration",SPLIT,holdout_delta_gini_and_shape,-0.536228,-2.451512,1.541885,0.249581,0.220000,0.277324,0.056748,NaN,False,grouping_bins,0.941762,
2,CHANNEL,miscal,True,WEAK,calibration,RECALIBRATE,rank_order_ok_calibration_fail,0.546061,2.496466,0.509352,0.166225,0.180000,0.164454,-0.229938,0.287487,False,grouping_bins,NaN,
3,CHANNEL,tiny,False,INCONCLUSIVE,insufficient_power,NONE,insufficient_power,0.829365,3.791668,0.769595,0.087908,0.006667,0.004086,0.065512,NaN,False,grouping_bins,NaN,


In [15]:
serial = evaluate_segments(df, meta.columns, gates, grouping=grouping, n_jobs=1)
print("serial == parallel:", serial.decisions.equals(result.decisions))

serial == parallel: True


### Matched approval-rate Gini

A segment scored on a different part of the risk spectrum is not comparable to the portfolio at face
value. When the approval-rate gap reaches `Gates.ar_gap_trigger`, both sides are re-cut to the
lower-AR anchor and Gini is recomputed there. Columns stay null when the gap does not trigger.

In [16]:
result.decisions[
    [
        "segment_value",
        "gini",
        "ar_segment",
        "ar_reference",
        "ar_gap",
        "ar_gap_triggered",
        "matched_ar",
        "matched_ar_anchor",
        "gini_at_matched_ar",
        "gini_reference_at_matched_ar",
        "gini_at_matched_ar_gap",
        "ar_artifact_suspected",
    ]
]

,segment_value,gini,ar_segment,ar_reference,ar_gap,ar_gap_triggered,matched_ar,matched_ar_anchor,gini_at_matched_ar,gini_reference_at_matched_ar,gini_at_matched_ar_gap,ar_artifact_suspected
0,core,0.524566,0.896740,0.849981,0.046759,False,NaN,NaN,NaN,NaN,NaN,False
1,inverted,-0.536228,0.906729,0.849981,0.056748,False,NaN,NaN,NaN,NaN,NaN,False
2,miscal,0.546061,0.620043,0.849981,-0.229938,True,0.620043,segment,0.287487,-0.020458,0.307946,False
3,tiny,0.829365,0.915493,0.849981,0.065512,False,NaN,NaN,NaN,NaN,NaN,False


### PSI, by variable type

Because a grouping was supplied, the model's characteristics are compared on the grouping's own bins
(`grouping_bins`). `x1_woe` is not in the grouping, so it falls back to portfolio-level decile edges
applied unchanged to the segment (`numeric_portfolio_deciles`). Missing values always form their own
bin and are reported separately.

In [17]:
result.characteristics.sort_values("psi", ascending=False)[
    [
        "segment_value",
        "feature",
        "psi",
        "psi_method",
        "psi_n_bins",
        "psi_worst_bin",
        "psi_missing_share_segment",
        "psi_out_of_range_share_segment",
        "rank_reversal",
    ]
].head(12)

,segment_value,feature,psi,psi_method,psi_n_bins,psi_worst_bin,psi_missing_share_segment,psi_out_of_range_share_segment,rank_reversal
15,tiny,x1_woe,0.040133,numeric_portfolio_deciles,11,"(0.250746, 0.528104]",0.0,0.0,False
13,tiny,x2,0.037110,grouping_bins,8,"(-0.184584, 0.174521]",0.0,0.0,False
12,tiny,x1,0.015725,grouping_bins,6,"(1.63489, inf)",0.0,0.0,False
14,tiny,cat,0.007487,grouping_bins,5,C,0.0,0.0,False
11,miscal,x1_woe,0.005616,numeric_portfolio_deciles,11,"(0.00394822, 0.250746]",0.0,0.0,False
7,inverted,x1_woe,0.004525,numeric_portfolio_deciles,11,"(-inf, -1.27169]",0.0,0.0,True
8,miscal,x1,0.002254,grouping_bins,6,"(1.63489, inf)",0.0,0.0,False
9,miscal,x2,0.002029,grouping_bins,8,"(1.62796, inf)",0.0,0.0,False
10,miscal,cat,0.001148,grouping_bins,5,B,0.0,0.0,False
5,inverted,x2,0.000868,grouping_bins,8,"(1.62796, inf)",0.0,0.0,False


### Refit performance and predictor stability

`SPLIT` needs both: a material holdout Gini gain with a positive lower CI bound and a better proper
score, **and** predictors that hold up across vintages. If the performance case is made but stability
is not, the action becomes `MONITOR`.

Too few vintages is **not** instability. `stability_summary` sets `stability_pass=True` with
`stability_reason="insufficient_vintages"` when nothing could be assessed (`assessed=False`,
`n_vintages` below `Gates.stability_min_vintages`, default 3). That outcome means "not judged", so it
does not veto an otherwise sound refit.

In [18]:
result.refit_comparison[
    [
        "segment_value",
        "n_holdout",
        "gini_pooled",
        "gini_refit",
        "delta_gini",
        "delta_gini_ci_low",
        "gini_recal",
        "refit_method",
        "stability_score",
        "stability_pass",
        "stability_reason",
    ]
]

,segment_value,n_holdout,gini_pooled,gini_refit,delta_gini,delta_gini_ci_low,gini_recal,refit_method,stability_score,stability_pass,stability_reason
0,core,1833.0,0.507007,NaN,NaN,NaN,NaN,NaN,NaN,True,not_attempted
1,inverted,729.0,-0.595251,0.497162,1.092413,0.987913,0.595251,logistic_woe,0.941762,True,stable
2,miscal,599.0,0.544438,NaN,NaN,NaN,0.544438,NaN,NaN,True,not_attempted
3,tiny,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,not_attempted


In [19]:
result.stability[
    [
        "segment_col",
        "segment_value",
        "feature",
        "n_vintages",
        "assessed",
        "psi_max",
        "gini_reference",
        "gini_ratio_median",
        "weak_vintage_share",
        "sign_consistency",
        "stability_score",
        "stable",
        "stability_flags",
    ]
]

,segment_col,segment_value,feature,n_vintages,assessed,psi_max,gini_reference,gini_ratio_median,weak_vintage_share,sign_consistency,stability_score,stable,stability_flags
0,__overall__,ALL,x1,18,True,0.009079,0.142835,1.033809,0.0,1.000000,0.987895,False,overlapping_event_rate_bounds
1,__overall__,ALL,x2,18,True,0.022288,0.209548,1.034616,0.0,1.000000,0.970282,False,overlapping_event_rate_bounds
2,__overall__,ALL,cat,18,True,0.013325,0.055642,1.099316,0.0,0.888889,0.945197,False,overlapping_event_rate_bounds
3,CHANNEL,inverted,x1,13,True,0.043679,0.463805,1.066630,0.0,1.000000,0.941762,True,
4,CHANNEL,inverted,x2,13,False,NaN,0.026262,NaN,NaN,NaN,NaN,True,no_reference_signal
5,CHANNEL,inverted,cat,13,False,NaN,0.028426,NaN,NaN,NaN,NaN,True,no_reference_signal


`insufficient_vintages` is a "not assessed" outcome, not a veto. When a segment (or a short-history
slice) has fewer than `Gates.stability_min_vintages` months that clear `min_rows`, each predictor is
flagged `insufficient_vintages` with `assessed=False`. `stability_summary` still returns
`stability_pass=True` so Q2 does not block a refit for lack of history.

The slice below keeps only the first two months of the analysis frame — below the default threshold
of 3 — so you can verify the pass / reason pair.

In [20]:
date_col = meta.columns.col_date
periods = pd.to_datetime(df[date_col]).dt.to_period("M")
keep = sorted(periods.dropna().unique())[:2]
short = df.loc[periods.isin(keep)].copy()
print("kept vintages:", [str(p) for p in keep], " rows:", len(short))
print("Gates.stability_min_vintages =", gates.stability_min_vintages)

short_stab = predictor_stability(
    short,
    list(meta.columns.cols_pred),
    meta.columns.col_target,
    date_col,
    gates,
    grouping=grouping,
)
print(
    short_stab[
        ["feature", "n_vintages", "assessed", "stable", "stability_flags"]
    ].to_string(index=False)
)
short_summary = stability_summary(short_stab, gates)
print("stability_pass:", short_summary["stability_pass"])
print("stability_reason:", short_summary["stability_reason"])
print("stability_assessed:", short_summary["stability_assessed"])
assert short_summary["stability_pass"] is True
assert short_summary["stability_reason"] == "insufficient_vintages"

kept vintages: ['2023-01', '2023-02']  rows: 1281
Gates.stability_min_vintages = 3
feature  n_vintages  assessed  stable       stability_flags
     x1           2     False    True insufficient_vintages
     x2           2     False    True insufficient_vintages
    cat           2     False    True insufficient_vintages
stability_pass: True
stability_reason: insufficient_vintages
stability_assessed: False


In [21]:
action_list(result)[["segment_value", "q1_verdict", "failed_pillars", "q2_action", "q2_reason"]]

,segment_value,q1_verdict,failed_pillars,q2_action,q2_reason
1,inverted,WEAK,"rank_order,calibration",SPLIT,holdout_delta_gini_and_shape
2,miscal,WEAK,calibration,RECALIBRATE,rank_order_ok_calibration_fail


### Segment vs portfolio grouping

A refit always fits **new** segment bins. The grouping passed into `evaluate_segments` is the
portfolio baseline. `result.grouping_comparison` notes per-bin edge shifts, merges/splits, WoE
shifts and sign flips. Significant notes are also copied onto the segment `BinSpec` so the saved
`grouping.json` carries the commentary.

In [22]:
cmp = result.grouping_comparison
print("comparison rows:", len(cmp))
if cmp.empty:
    show = cmp
else:
    print("kinds:", dict((k, int(v)) for k, v in cmp["kind"].value_counts().items()))
    print("significant:", int(cmp["significant"].sum()))
    show = cmp.loc[cmp["significant"]] if cmp["significant"].any() else cmp
show[
    ["feature", "segment_bin", "portfolio_bins", "woe_segment", "woe_portfolio", "woe_delta", "kind", "note"]
].head(12)

comparison rows: 8
kinds: {'merged+woe_sign_flip': 2, 'split': 2, 'woe_sign_flip': 1, 'merged+woe_shift': 1, 'membership_change': 1, 'aligned': 1}
significant: 7


,feature,segment_bin,portfolio_bins,woe_segment,woe_portfolio,woe_delta,kind,note
0,x1,"(-inf, 0.144754]","(-inf, 0.544101]",-0.696349,0.189597,-0.885946,woe_sign_flip,"x1 bin (-inf, 0.144754] WoE sign flipped vs po..."
1,x1,"(0.144754, inf)","(-inf, 0.544101],(0.544101, 1.12006],(1.12006,...",1.940132,-1.019020,2.959152,merged+woe_sign_flip,"x1 bin (0.144754, inf) spans portfolio bins [(..."
2,x1,"(-inf, 0.144754],(0.144754, inf)","(-inf, 0.544101]",NaN,0.189597,NaN,split,"x1 portfolio bin (-inf, 0.544101] is split acr..."
3,x2,"(-inf, 1.38421]","(-inf, -0.998325],(-0.998325, -0.184584],(-0.1...",-0.027242,0.701982,-0.729223,merged+woe_shift,"x2 bin (-inf, 1.38421] spans portfolio bins [(..."
4,x2,"(1.38421, inf)","(0.957649, 1.62796],(1.62796, inf)",0.386130,-0.787588,1.173718,merged+woe_sign_flip,"x2 bin (1.38421, inf) spans portfolio bins [(0..."
5,x2,"(-inf, 1.38421],(1.38421, inf)","(0.957649, 1.62796]",NaN,-0.552520,NaN,split,"x2 portfolio bin (0.957649, 1.62796] is split ..."
6,cat,A|B,"A,B",-0.034643,0.030522,-0.065165,membership_change,"cat bin A|B mixes portfolio groups [A,B] (WoE ..."


### Fitted model artefacts

Refit and recalibration each return a `FittedModelArtifact` (the grouping plus the estimator) so
scores can be reproduced later. `result.save_artifacts` writes one directory per segment and kind
containing `grouping.json`, `model.pkl` and `meta.json`. Logistic refits also write `scorecard.sql`
in the same CASE WHEN + `LINEAR_SCORE` shape as `tests/fixtures/sample_scorecard.sql`.

In [23]:
artifact_dir = os.path.join(WORKDIR, "fitted")
written = result.save_artifacts(artifact_dir)
print("wrote %d artefact directories" % (len(written),))
for path in written:
    names = sorted(os.listdir(path))
    print("  ", path)
    print("     ", ", ".join(names))

if written:
    art = load_fitted_artifact(written[0])
    print("reloaded kind=%s method=%s features=%s" % (art.kind, art.method, art.feature_names))
    print("has grouping:", art.grouping is not None, "| has model:", art.model is not None)
    sql_files = [os.path.join(p, "scorecard.sql") for p in written if os.path.isfile(os.path.join(p, "scorecard.sql"))]
    print("scorecard.sql files:", len(sql_files))
    if sql_files:
        with open(sql_files[0]) as handle:
            preview = handle.read().splitlines()[:12]
        print("\n".join(preview))

wrote 3 artefact directories
   /workspace/notebooks/demo_output/fitted/CHANNEL/inverted/recalibrate
      grouping.json, meta.json, model.pkl
   /workspace/notebooks/demo_output/fitted/CHANNEL/inverted/refit
      grouping.json, grouping_comparison.json, meta.json, model.pkl, scorecard.sql
   /workspace/notebooks/demo_output/fitted/CHANNEL/miscal/recalibrate
      grouping.json, meta.json, model.pkl
reloaded kind=recalibrate method=logistic_pd features=['logit_pd']
has grouping: True | has model: True
scorecard.sql files: 1
select
1/(1+exp(-s.LINEAR_SCORE)) as SCORE,
s.*
from (
    select
    w.x1_woe * -1.0054629111791444
     + w.x2_WOE * -0.86532132210362922
     + w.cat_WOE * -1.333755052075972
     + w.Intercept * -1.1888742653933242
    as LINEAR_SCORE,
    w.*
    from (


### The optional XGBoost sub-model

`submodel=True` fits an XGBoost pillar model with `max_depth` hard-capped at 3, so at most three-way
interactions. Without `xgboost` installed it warns clearly and falls back to the logistic refit rather
than raising, which is why this cell is safe to run either way.

In [24]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    submodel_result = evaluate_segments(
        df, meta.columns, gates, grouping=grouping, submodel=True
    )

for warning in caught:
    if "xgboost" in str(warning.message):
        print("WARNING:", warning.message)
        break

submodel_result.refit_comparison[
    ["segment_value", "refit_method", "gini_pooled", "gini_refit", "delta_gini"]
]

,segment_value,refit_method,gini_pooled,gini_refit,delta_gini
0,core,NaN,0.507007,NaN,NaN
1,inverted,logistic_woe,-0.595251,0.497162,1.092413
2,miscal,NaN,0.544438,NaN,NaN
3,tiny,NaN,NaN,NaN,NaN


## 6. Production scorecard SQL

A logistic scorecard shipped as SQL is enough to reconstruct the pooled model: CASE WHEN WoE bins,
`nvl(LN(p/(1-p)), impute)` logit / VAL columns, then `LINEAR_SCORE = B^T X` folded through a sigmoid.

The sample at `tests/fixtures/sample_scorecard.sql` is the shape production platforms emit. Names need
not match (`indosat_v2` maps to `feature_a_WOE`). Numeric `WHEN x < t` / `x >= t` chains are left-closed
`[a, b)` bins. Null and else branches are stored as `impute` / `missing_note`.

`render_scorecard_sql` writes that same shape from a grouping and coefficients; logistic refit
artefacts include the generated `scorecard.sql`.

In [25]:
parsed = parse_scorecard_sql_path(SQL_PATH)
print("cols_pred:     ", parsed.cols_pred)
print("cols_pred_woe: ", parsed.cols_pred_woe)
print("cols_pred_used:", parsed.cols_pred_used)
print("pred_map:      ", parsed.pred_map)
print()
print(parsed.formula)

cols_pred:      ['indosat_v2', 'featureB', 'featC', 'feat_pred_D', 'featE', 'featF_v3_0', 'featG_V2', 'featH_v4_0', 'feati_v3', 'var_v2']
cols_pred_woe:  ['feature_a_WOE', 'featureB_WOE', 'featC_WOE', 'feat_pred_D_WOE']
cols_pred_used: ['feature_a_WOE', 'featureB_WOE', 'featC_WOE', 'feat_pred_D_WOE', 'featE_VAL', 'featF_v3_0_VAL', 'featG_V2_VAL', 'featH_v4_0_VAL', 'feati_v3_VAL', 'var_v2_VAL']
pred_map:       {'indosat_v2': 'feature_a_WOE', 'featureB': 'featureB_WOE', 'featC': 'featC_WOE', 'feat_pred_D': 'feat_pred_D_WOE', 'featE': 'featE_VAL', 'featF_v3_0': 'featF_v3_0_VAL', 'featG_V2': 'featG_V2_VAL', 'featH_v4_0': 'featH_v4_0_VAL', 'feati_v3': 'feati_v3_VAL', 'var_v2': 'var_v2_VAL'}

PD = 1/(1+exp(-LINEAR_SCORE))
LINEAR_SCORE = B^T X
X = [feature_a_WOE, featureB_WOE, featC_WOE, feat_pred_D_WOE, featE_VAL, featF_v3_0_VAL, featG_V2_VAL, featH_v4_0_VAL, feati_v3_VAL, var_v2_VAL, 1]
B = [feature_a_WOE, featureB_WOE, featC_WOE, feat_pred_D_WOE, featE_VAL, featF_v3_0_VAL, featG_V2_VAL, fe

In [26]:
sql_grouping_path = os.path.join(WORKDIR, "sql_grouping.json")
parsed.save_grouping(sql_grouping_path)
impute_rows = []
for name in parsed.grouping.columns:
    spec = parsed.grouping.specs[name]
    impute_rows.append(
        {
            "feature": name,
            "output_name": spec.output_name,
            "kind": spec.kind,
            "closed": spec.closed,
            "impute": spec.impute,
            "missing_note": "; ".join(spec.notes),
        }
    )
print("wrote", sql_grouping_path)
pd.DataFrame(impute_rows)

wrote /workspace/notebooks/demo_output/sql_grouping.json


,feature,output_name,kind,closed,impute,missing_note
0,indosat_v2,feature_a_WOE,numeric,left,-0.008113,null_impute: -0.008113478679658392
1,featureB,featureB_WOE,mixed,left,-0.198029,null_impute: -0.19802888585164036
2,featC,featC_WOE,numeric,left,0.049657,null_impute: 0.04965688662575474
3,feat_pred_D,feat_pred_D_WOE,categorical,right,0.095775,null_impute: 0.09577533753541978
4,featE,featE_VAL,logit,right,-2.701125,null_impute: -2.701124677318522
5,featF_v3_0,featF_v3_0_VAL,logit,right,NaN,no_null_imputation
6,featG_V2,featG_V2_VAL,logit,right,-2.975533,null_impute: -2.97553316366986
7,featH_v4_0,featH_v4_0_VAL,logit,right,-2.730217,null_impute: -2.730217
8,feati_v3,feati_v3_VAL,logit,right,-2.557911,null_impute: -2.557910899788164
9,var_v2,var_v2_VAL,logit,right,-2.735600,null_impute: -2.7356004290961025


In [27]:
toy = pd.DataFrame(
    {
        "indosat_v2": [0.01, 0.030322997830808163, float("nan")],
        "featureB": [600.0, "No Score", float("nan")],
        "featC": [0.05, 0.2, float("nan")],
        "feat_pred_D": ["0-1month", None, "unseen-bucket"],
        "featE": [0.5, float("nan"), 0.8],
        "featF_v3_0": [0.5, 0.5, 0.5],
        "featG_V2": [0.5, 0.5, 0.5],
        "featH_v4_0": [0.5, 0.5, 0.5],
        "feati_v3": [0.5, 0.5, 0.5],
        "var_v2": [0.5, 0.5, 0.5],
    }
)
mapped = parsed.grouping.transform_woe(toy)
pd_hat = parsed.predict_proba(toy)[:, 1]
sklearn_pd = parsed.model.predict_proba(parsed.grouping.transform(toy))[:, 1]
print("sklearn intercept_ = %.6f" % (float(parsed.model.intercept_[0]),))
print("max |parse - sklearn| = %.3g" % (abs(pd_hat - sklearn_pd).max(),))
scored = mapped.copy()
scored["PD"] = pd_hat
scored

sklearn intercept_ = 7.864808
max |parse - sklearn| = 0


,indosat_v2,featureB,featC,feat_pred_D,featE,featF_v3_0,featG_V2,featH_v4_0,feati_v3,var_v2,PD
0,0.858988,0.350913,-0.319831,-0.374421,0.000000,0.0,0.0,0.0,0.0,0.0,0.999524
1,0.145065,-0.198029,-0.940852,0.095775,-2.701125,0.0,0.0,0.0,0.0,0.0,0.999388
2,-0.008113,-0.198029,0.049657,3.660287,1.386294,0.0,0.0,0.0,0.0,0.0,0.997483


In [28]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    sql_meta = resolve_metadata(
        table="risk.scorecard_base",
        col_id="SKP_CREDIT_CASE",
        col_score="SCORE",
        model_sql_path=SQL_PATH,
        grouping_path=sql_grouping_path,
        executor=executor,
    )

print("inferred from SQL:")
for field in ("cols_pred", "cols_pred_woe", "cols_pred_used", "grouping_path"):
    print("  %-16s %s" % (field + ":", sql_meta.sources.get(field)))
print("pred_woe_map:   ", sql_meta.columns.pred_woe_map())
print("psi_grouping:  ", "psi_grouping" in sql_meta.capabilities.available)
print("formula starts:", (sql_meta.formula or "").splitlines()[0])

inferred from SQL:
  cols_pred:       inferred:sql_model
  cols_pred_woe:   inferred:sql_model
  cols_pred_used:  inferred:sql_model
  grouping_path:   supplied
pred_woe_map:    {'indosat_v2': 'feature_a_WOE', 'featureB': 'featureB_WOE', 'featC': 'featC_WOE', 'feat_pred_D': 'feat_pred_D_WOE'}
psi_grouping:   True
formula starts: PD = 1/(1+exp(-LINEAR_SCORE))


## 7. The final report

`recommendations` turns the evaluation into a prioritised, concrete action list, and `save_report`
writes a self-contained HTML page (no external assets, no extra dependencies) with the per-segment
verdicts, the matched-AR comparison, the PSI method used, the grouping comparison, and both the
refit performance and the stability findings.

In [29]:
recs = recommendations(result, gates)
recs[["rank", "priority", "segment_value", "action", "recommendation"]]

,rank,priority,segment_value,action,recommendation
0,1,0,inverted,SPLIT,Split out a dedicated model for CHANNEL = inve...
1,2,1,miscal,RECALIBRATE,Recalibrate the PD level for CHANNEL = miscal
2,3,2,ALL,MONITOR,"Stabilise portfolio-level predictors: x1, x2, cat"
3,4,5,tiny,NONE,Do not judge CHANNEL = tiny yet -- insufficien...


In [30]:
for _index, row in recs.iterrows():
    print("%d. [%s] %s" % (row["rank"], row["action"], row["recommendation"]))
    print("   why:      " + row["why"])
    print("   evidence: " + row["evidence"])
    print()

1. [SPLIT] Split out a dedicated model for CHANNEL = inverted
   why:      The pooled score fails rank ordering here, the WoE shape diverges from the portfolio, and a same-predictor refit beats the pooled score on the forward holdout by 1.092 Gini (lower CI bound 0.988) with a better proper score. Predictor stability over vintages also clears the gate, so the new coefficients are expected to hold.
   evidence: Gini -0.536 (ratio to portfolio -2.45); O/E 1.54, ECE 0.250; volume share 22.0%, default share 27.7%; PSI method: grouping_bins; predictor stability score 0.94

2. [RECALIBRATE] Recalibrate the PD level for CHANNEL = miscal
   why:      Ranking is intact but the PD level is off (O/E 0.51, ECE 0.166). A two-parameter intercept/slope rescale fixes the level without touching the ranking, so no split is warranted.
   evidence: Gini 0.546 (ratio to portfolio 2.50); O/E 0.51, ECE 0.166; volume share 18.0%, default share 16.4%; approval-rate gap -23.0%, Gini at matched AR 0.287 (portfol

In [31]:
html_path = save_report(
    result,
    os.path.join(WORKDIR, "segment_evaluation_report.html"),
    title="Segment scorecard evaluation -- synthetic book",
    gates=gates,
)
md_path = save_report(
    result,
    os.path.join(WORKDIR, "segment_evaluation_report.md"),
    title="Segment scorecard evaluation -- synthetic book",
    gates=gates,
)
print("HTML report:    %s (%d bytes)" % (html_path, os.path.getsize(html_path)))
print("Markdown report: %s (%d bytes)" % (md_path, os.path.getsize(md_path)))

HTML report:    /workspace/notebooks/demo_output/segment_evaluation_report.html (36144 bytes)
Markdown report: /workspace/notebooks/demo_output/segment_evaluation_report.md (6858 bytes)


In [32]:
from IPython.display import IFrame

IFrame(src=os.path.relpath(html_path, os.getcwd()), width="100%", height=650)

## Recap

| Step | Output |
| --- | --- |
| Metadata resolution | inferred date, segmentation and target/observation pair, each with a warning |
| Capability resolution | which of the nine analyses can run, and why the rest cannot |
| Confirmation | settings persisted and reloaded losslessly |
| WoE map | `cols_pred` → `cols_pred_woe` (`x1` → `x1_woe`; SQL aliases win when names differ) |
| Grouping | supervised WoE bins serialised to `grouping.json` |
| Evaluation | Q1 verdict, Q2 action, matched-AR Gini, type-aware PSI, predictor stability |
| Grouping comparison | per-bin edge / WoE / sign-flip notes vs the portfolio grouping |
| Fitted artefacts | `grouping.json` + estimator + `scorecard.sql` per logistic refit, reloadable |
| Scorecard SQL | `cols_pred` / `_woe` / `_used`, `B^T X` formula, null imputation, sklearn LR |
| Report | prioritised recommendations plus a self-contained HTML page |